# v4 demo -- automotive complaint safety triage

Runs on **Colab or Kaggle free-tier GPU** (same as every training/eval notebook in this
project). Paste a raw complaint, get structured JSON back.

## Why a notebook, not a local app

Phase 4 (merge the adapter to 16-bit, quantize to GGUF, run it locally via Ollama) is
**formally descoped** -- the merge+conversion pipeline needs the merged model (~16GB)
and the F16 intermediate (~16.4GB) to coexist on disk at once, ~32GB peak. That exceeds
Kaggle's fixed 20GB disk limit (confirmed twice) and OOM-killed Colab's from-source
llama.cpp build even after capping parallel jobs (confirmed once). A structural
constraint of the pipeline, not something more retries would fix -- see
`docs/blueprint.md` Section 6a.

This notebook sidesteps all of that: it loads the base model + the v4 LoRA/DoRA adapter
**in 4-bit, no merge, no GGUF conversion** -- the exact same pattern already proven
working across every training and eval run in this project
(`notebooks/eval_baseline_vs_finetuned_v4.ipynb`).

**This is a demonstration, not the project's evidence base.** For the real accuracy
numbers, error analysis, and honest limitations, see
[`docs/eval-report.md`](../docs/eval-report.md).

## 1. Install dependencies (Colab/Kaggle only)

In [ ]:
%%capture
!pip install unsloth


## 2. Load the v4 adapter (base model + LoRA/DoRA, 4-bit)

Upload checklist -- bundle these 6 files from
`models/qwen3-8b-automotive-complaint-lora-v4-FINAL/` into one Kaggle Dataset (any name
works, auto-detected) or upload individually on Colab -- same files already used for
the last eval run:
`adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`,
`tokenizer_config.json`, `chat_template.jinja`, `README.md`.

In [ ]:
import glob, os

def find_kaggle_dataset(*required_files):
    """Search /kaggle/input/*/ for a folder containing all of required_files --
    works no matter what the attached Dataset is named, so re-uploading under a new
    name never requires editing this notebook."""
    for candidate in sorted(glob.glob("/kaggle/input/*/")):
        if all(os.path.exists(os.path.join(candidate, f)) for f in required_files):
            return candidate.rstrip("/")
    return None

ADAPTER_DIR = find_kaggle_dataset("adapter_config.json", "adapter_model.safetensors")
if ADAPTER_DIR is None:
    try:
        from google.colab import files
        print("Upload the 6 adapter files listed above:")
        files.upload()
        ADAPTER_DIR = "."
    except ImportError:
        raise RuntimeError(
            "No /kaggle/input/*/ folder found containing adapter_config.json + "
            "adapter_model.safetensors. Attach a Dataset with the adapter's inference "
            "files in it -- any dataset name works, no path editing needed."
        )
print(f"using adapter from: {ADAPTER_DIR}")

MAX_SEQ_LENGTH = 896  # matches train_qlora_dora_v4.ipynb / eval_baseline_vs_finetuned_v4.ipynb

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)
FastLanguageModel.for_inference(model)  # Unsloth's native 2x-faster inference mode
print("model + v4 adapter loaded")


## 3. Prompt format (matches training exactly)

`SYSTEM_PROMPT` copied verbatim from `notebooks/train_qlora_dora_v4.ipynb` and
`notebooks/eval_baseline_vs_finetuned_v4.ipynb` -- must match training exactly, same
requirement as the eval notebook.

In [ ]:
import json
import re

SYSTEM_PROMPT = (
    "You are an automotive safety complaint analyst. Given a raw consumer complaint "
    "about a vehicle, extract a structured JSON object with exactly these fields: "
    'component (string), defect_type (string), safety_risk ("yes" or "no"), '
    'severity ("low", "medium", or "high"), crash_described (true or false -- does '
    'the complaint text itself describe an actual collision/impact, not just '
    'mention a safety feature by name), fire_described (true or false -- does the '
    'text describe an actual fire/smoke/explosion event), injury_described (true '
    'or false -- does the text describe an actual injury to a person, not a '
    'hypothetical or averted one). Respond with only the JSON object.'
)

MAX_NEW_TOKENS = 130  # 7-field JSON; generous headroom, matches the eval notebook

def build_prompt(tokenizer, narrative):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Complaint:\n{narrative}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

_JSON_OBJ_PATTERN = re.compile(r"\{.*?\}", re.DOTALL)

def parse_json_output(raw_text):
    """Extract and parse the first {...} block. Returns (parsed_dict_or_None, raw_text).
    Never raises -- a model going off-script (extra prose, missing braces, etc.) is
    something to show the user, not a notebook crash."""
    match = _JSON_OBJ_PATTERN.search(raw_text)
    if not match:
        return None, raw_text
    try:
        obj = json.loads(match.group(0))
        return (obj, raw_text) if isinstance(obj, dict) else (None, raw_text)
    except json.JSONDecodeError:
        return None, raw_text


## 4. Generation function

In [ ]:
import torch

def analyze(narrative):
    """Runs the model on one complaint narrative. Returns (parsed_dict_or_None, raw_text)."""
    prompt_text = build_prompt(tokenizer, narrative)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy -- deterministic, matches every eval run
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_json_output(raw_output)


## 5. Example complaints -- edit `NARRATIVE` and rerun

Four real NHTSA complaints from the eval set, spanning the label range, so this is
demo-ready without writing examples on the spot. Swap `NARRATIVE` for anything else --
your own text works too, it doesn't need to be a real NHTSA complaint.

In [ ]:
EXAMPLE_COMPLAINTS = {
    # odino=550381 -- actual: safety_risk=no, severity=low
    "low_no_risk": (
        "DRIVER AND PASSENGER SIDE PLASTIC DOOR LOCK COVERS BROKE, CONSUMER HAD THEM "
        "REPLACED WITH METAL ONES TO PREVENT FUTURE BREAKAGE.  NLM"
    ),
    # odino=10444327 -- actual: safety_risk=no, severity=low (dramatic-sounding but
    # nothing actually happened -- a good check on whether the model over-flags)
    "low_hypothetical": (
        "TL* THE CONTACT OWNS A 2003 TOYOTA 4RUNNER. THE CONTACT STATED THAT THE "
        "DASHBOARD WAS CRACKED FROM THE DRIVER TO THE PASSENGER SIDE AIR BAG AREAS. "
        "THE CONTACT WAS CONCERNED THAT THE AIR BAGS WOULD DEPLOY. THE VEHICLE WAS "
        "NOT REPAIRED. THE FAILURE MILEAGE WAS 115,000 AND THE CURRENT MILEAGE WAS "
        "118,364.  UPDATED 02/28/12 *BF"
    ),
    # odino=10949873 -- actual: safety_risk=yes, severity=medium
    "medium_fire": (
        "MY DRIVER SIDE SEAT HEATER CAUGHT ON FIRE TODAY AND SMOLDERED A HOLE "
        "THROUGH THE SEAT.  I WAS DRIVING IN A RESIDENTIAL NEIGHBORHOOD IN THE "
        "MORNING AROUND 9:15AM."
    ),
    # odino=981275 -- actual: safety_risk=yes, severity=high
    "high_crash": (
        "FRONTAL COLLISION, HIT A DEER AT SPEED OF 30MPH, IMPACT 12:00 POSITION, "
        "DUAL AIR BAGS DID NOT DEPLOY.  *AK"
    ),
}

NARRATIVE = EXAMPLE_COMPLAINTS["medium_fire"]  # <-- EDIT THIS, then rerun the cell below


## 6. Run it

In [ ]:
parsed, raw_output = analyze(NARRATIVE)

print("complaint:")
print(f"  {NARRATIVE}\n")

if parsed is None:
    print("Model output did not parse as valid JSON -- raw output:")
    print(raw_output)
else:
    print("extracted:")
    print(json.dumps(parsed, indent=2))
